In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from config_train import get_config
from torch.utils.data import DataLoader
from tsp import *
from cvrp import CVRPDataset
import torch
import numpy as np
import time
import datetime
import tqdm
import os
import logging
import sys
from utils import read_instance_data
# import importlib
# importlib.reload(sys.modules['search_control'])
# importlib.reload(sys.modules['de'])
from search_control import solve_instance_set
from VAE_8 import VAE_8
torch.set_float32_matmul_precision('high')
from torch.utils.tensorboard import SummaryWriter

def calculate_Reg_loss(pred_costs,instances,solutions):
    true_cost = tours_length(instances,solutions).reshape(pred_costs.shape)

    Reg = torch.nn.functional.mse_loss(pred_costs, true_cost)
    return Reg


def calculate_KLD_loss(mean, log_var):
    KLD = -0.5 * torch.sum(1 + log_var - mean.pow(2) - log_var.exp()) / 2
    return KLD

def calculate_RC_loss(tour_logp):
    RC = - tour_logp.sum() / 2
    return RC


In [3]:
def evaluate_network(config, model, validation_dataloader, epoch_idx):
    model.eval()
    loss_RC_values = []
    loss_KLD_values = []
    loss_REG_values = []
    abs_Z_values = []
    for batch_id, batch in enumerate(validation_dataloader):
        instances, solutions_1, solutions_2 = batch

        with torch.no_grad():
            output, mean, log_var, Z, tour_idx, tour_logp, pred_costs = model(instances, solutions_1, solutions_2, config)
        loss_RC = calculate_RC_loss(tour_logp)
        loss_KLD = calculate_KLD_loss(mean, log_var)
        loss_REG = calculate_Reg_loss(pred_costs,instances,solutions_1)



        loss_RC_values.append(loss_RC.item())
        loss_KLD_values.append(loss_KLD.item())
        loss_REG_values.append(loss_REG.item())
        abs_Z = torch.abs(Z)  # Absolute coordinates of points in latent space (Z)
        abs_Z_values.append(abs_Z.cpu().numpy())

    abs_Z_values = np.array(abs_Z_values).flatten()

    # The bounds of the search space are defined as a percentile of the absolute latent variable coordinates
    new_bound = np.percentile(abs_Z_values, config.q_percentile).item()

    avgRC = np.mean(loss_RC_values)
    avgKL = np.mean(loss_KLD_values)
    avgREG = np.mean(loss_REG_values)
    writer.add_scalar("Loss/Reconstruction/Val", avgRC, epoch_idx)
    writer.add_scalar("Loss/KL-divergence/Val", avgKL, epoch_idx)
    writer.add_scalar("Loss/Regression/Val", avgREG, epoch_idx)
    writer.add_scalar("Loss/Combined/Val", avgRC + config.KLD_weight * avgKL, epoch_idx)

    return new_bound

In [4]:
VERSION = "0.4.0"
run_id = np.random.randint(10000, 99999)
now = datetime.datetime.now()

config = get_config(inJupyter=True)

if config.output_path == "":
    config.output_path = os.getcwd()
run_id = f"run_{now.day}.{now.month}_{now.hour}-{now.minute}-{now.second}_{run_id}"
config.output_path = os.path.join(config.output_path, "runs", run_id)
os.makedirs(os.path.join(config.output_path, "models"))

writer = SummaryWriter(log_dir=f"tensorboard_logdir/{run_id}")
writer.add_text('config', '\n'.join(map(lambda k_v: f"{k_v[0]}: {k_v[1]}", vars(config).items())))

logging.basicConfig(
    filename=os.path.join(config.output_path, "log_" + str(run_id) + ".txt"), filemode='w',
    level=logging.INFO, format='[%(levelname)s]%(message)s', force=True)
logging.info("Started Training Run")
logging.info("Call: {0}".format(''.join(sys.argv)))
logging.info("Version: {0}".format(VERSION))
logging.info("PARAMETERS:")
for arg in sorted(vars(config)):
    logging.info("{0}: {1}".format(arg, getattr(config, arg)))
logging.info("----------")
training_data, validation_data = read_instance_data(config) # this takes 10 fucking seconds...

In [5]:
def train_step(instances, solutions_1, solutions_2):
    optimizer.zero_grad()
    output, mean, log_var, Z, tour_idx, tour_logp, pred_costs = model(instances, solutions_1, solutions_2, config)

    loss_Reg = calculate_Reg_loss(pred_costs,instances,solutions_1)
    loss_RC = calculate_RC_loss(tour_logp)
    loss_KLD = calculate_KLD_loss(mean, log_var)
    REG_weight = 1000

    loss = loss_RC + loss_KLD * config.KLD_weight + loss_Reg*REG_weight
    assert not torch.isnan(loss)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
    optimizer.step()
    return loss_RC, loss_KLD, loss_Reg


def train_epoch(epoch_idx):
    global training_instances, training_solutions
    model.train()
    # n = training_instances.shape[0]
    # perm = torch.randperm(n)
    # training_instances = training_instances[perm, :]
    # training_solutions = training_solutions[perm, :]
    # shift1 = torch.randint(high=config.problem_size, size=(1,))
    # s1 = torch.roll(training_solutions, int(shift1), 1)
    # shift2 = torch.randint(high=config.problem_size, size=(1,))
    # s2 = torch.roll(training_solutions, int(shift2), 1)

    # for batch_id in range((training_instances.shape[0] + config.batch_size - 1) // config.batch_size):
    #     sl = slice(batch_id * config.batch_size, (batch_id + 1) * config.batch_size)
    #     instances, solutions_1, solutions_2 = training_instances[sl], s1[sl], s2[sl]
    loss_RC_values = []
    loss_KLD_values = []
    loss_REG_values = []
    for batch_id, batch in enumerate(training_dataloader):
        instances, solutions_1, solutions_2 = batch
        loss_RC, loss_KLD, loss_Reg= train_step(instances, solutions_1, solutions_2)
        loss_RC_values.append(loss_RC.item())
        loss_KLD_values.append(loss_KLD.item())
        loss_REG_values.append(loss_Reg.item())
        lr_scheduler.step()
    avgRC = np.mean(loss_RC_values)
    avgKL = np.mean(loss_KLD_values)
    avgREG = np.mean(loss_REG_values)
    writer.add_scalar("Loss/Reconstruction/Train", avgRC, epoch_idx)
    writer.add_scalar("Loss/KL-divergence/Train", avgKL, epoch_idx)
    writer.add_scalar("Loss/Regression/Train", avgREG, epoch_idx)
    writer.add_scalar("Loss/Combined/Train", avgRC + config.KLD_weight * avgKL, epoch_idx)
    writer.add_scalar("LR", lr_scheduler.get_last_lr()[0], epoch_idx)

In [110]:
def do(starting_epoch = 0):
    best_avg_gap = np.inf
    for epoch_idx in tqdm.trange(1, config.nb_epochs + 1):
        epoch_idx += starting_epoch
        train_epoch(epoch_idx)
        if epoch_idx % 50 == 0:
            new_bound = evaluate_network(config, model, validation_dataloader, epoch_idx)

            config.search_space_bound = new_bound
            writer.add_scalar("Search Space Bounds", new_bound, epoch_idx)

            avg_gap, avg_runtime, _ = solve_instance_set(model, config,
                                                         validation_data[0][: config.search_validation_size]
                                                         , validation_data[1][:config.search_validation_size])

            # If the average gap is improved, save the model
            if avg_gap < best_avg_gap:
                best_avg_gap = avg_gap
                model_data = {
                    'parameters': model.state_dict(),
                    'code_version': VERSION,
                    'problem': config.problem,
                    'problem_size': config.problem_size,
                    'Z_bound': new_bound,
                    'avg_gap': avg_gap,
                    'training_epochs': epoch_idx,
                    'model': "VAE_final"
                }

                torch.save(model_data, os.path.join(config.output_path, "models",
                                                    "model_{0}.pt".format(run_id, epoch_idx)))

            writer.add_scalar("Gap/Val", avg_gap * 100, epoch_idx)
            writer.add_scalar("Search Time/Val", avg_runtime, epoch_idx)
    return best_avg_gap

In [116]:
from VAE_8 import VAE_8
model = VAE_8(config).to(config.device)
optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)
# training_data = torch.load('training_data.torch')
# training_data[0] = training_data[0].numpy()
# training_data[1] = training_data[1].numpy()

In [112]:
# validation_data = torch.load('validation_data.torch')
# validation_data[0] = validation_data[0].numpy()
# validation_data[1] = validation_data[1].numpy()
training_dataset = TSPDataset(config.epoch_size, config.problem_size, config, training_data)
training_dataloader = DataLoader(training_dataset, batch_size=config.batch_size, num_workers=0, shuffle=True)
validation_dataset = TSPDataset(config.network_validation_size, config.problem_size, config, validation_data)
validation_dataloader = DataLoader(validation_dataset, batch_size=config.batch_size, num_workers=0, shuffle=True)
lr_scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, config.lr, epochs=config.nb_epochs, steps_per_epoch=len(training_dataloader))
# training_instances, training_solutions = training_data
# training_instances = training_instances.float().to(config.device)
# training_solutions = training_solutions.to(config.device)

In [113]:
avg_gap = do()
writer.add_hparams(config, {"hparam/avg_gap": avg_gap * 100})

  8%|▊         | 49/600 [08:54<1:27:23,  9.52s/it]

Costs 3.769585132598877
Costs 3.874239206314087
Costs 4.004540920257568
Costs 3.816547393798828
Costs 4.495426177978516
Costs 3.7865636348724365
Costs 3.8831405639648438
Costs 4.202723503112793
Costs 3.658531665802002
Costs 3.3524510860443115
Costs 4.142167568206787
Costs 4.302555561065674
Costs 3.9417924880981445
Costs 3.96234393119812
Costs 4.032354354858398
Costs 4.2521891593933105
Costs 3.4681451320648193
Costs 3.960942506790161
Costs 4.0115509033203125
Costs 3.405766248703003
Costs 3.0125346183776855
Costs 3.227724075317383
Costs 3.8299667835235596
Costs 3.7230567932128906
Costs 3.2888383865356445
Costs 3.7065060138702393
Costs 3.7934303283691406
Costs 4.186741352081299
Costs 3.794935703277588
Costs 3.7640867233276367
Costs 3.9694743156433105
Costs 3.8344898223876953
Costs 3.5392379760742188
Costs 3.79575514793396
Costs 3.4262185096740723
Costs 3.7623095512390137
Costs 4.060541152954102
Costs 3.897855520248413
Costs 3.219214677810669
Costs 4.259589195251465
Costs 3.747296094894409

  8%|▊         | 50/600 [17:44<25:19:42, 165.79s/it]

Costs 4.067282199859619


 16%|█▋        | 99/600 [26:24<1:32:33, 11.08s/it]  

...

100%|█████████▉| 599/600 [3:20:26<00:11, 11.17s/it]   

Costs 3.769585132598877
Costs 3.874239206314087
Costs 4.004540920257568
Costs 3.816547393798828
Costs 4.495426177978516
Costs 3.7865636348724365
Costs 3.8831405639648438
Costs 4.202723503112793
Costs 3.658531665802002
Costs 3.3524510860443115
Costs 4.142167568206787
Costs 4.2312211990356445
Costs 3.9417924880981445
Costs 3.96234393119812
Costs 4.032354354858398
Costs 4.2521891593933105
Costs 3.4681451320648193
Costs 3.960942506790161
Costs 4.0115509033203125
Costs 3.405766248703003
Costs 3.0125346183776855
Costs 3.227724075317383
Costs 3.8299667835235596
Costs 3.7230567932128906
Costs 3.2888383865356445
Costs 3.7065060138702393
Costs 3.7934303283691406
Costs 4.186741352081299
Costs 3.794935703277588
Costs 3.7640867233276367
Costs 3.9694743156433105
Costs 3.8344898223876953
Costs 3.5392379760742188
Costs 3.79575514793396
Costs 3.4262185096740723
Costs 3.7623095512390137
Costs 4.060541152954102
Costs 3.897855520248413
Costs 3.219214677810669
Costs 4.259589195251465
Costs 3.74729609489440

100%|██████████| 600/600 [3:29:20<00:00, 20.93s/it] 

Costs 4.067282199859619


TypeError: hparam_dict and metric_dict should be dictionary.

In [ ]:
model.decoder.transformer_decoder.logit_clipping